# Switching acquisition functions within a run: how much is there to gain?

## Description

**Question.** An amortized acquisition *policy* would pick the acquisition function (arm) at each stage of a BO run from the current state. That only pays off if some state-dependent choice of arms beats the best single arm. **How much better than the tuned best static arm B\* can a non-anticipating switching rule get, on tasks from the PFN's own prior?** If even a rule with privileged knowledge of the task gains little, a learned policy can't gain more, and the idea is not worth amortizing.

**Why a new design.** The predecessor's kill test (pfn-acqpolicy, notebooks 02–05) measured a *hindsight oracle*: the best of 81 branch sequences per task. That number was mostly luck. A null tree whose branches all ran the same arm, with only the nuisance differing, already reached 81–95% of the oracle's gain. The upper bound that survived was ≲ 5% (§16 there). The design here removes that selection effect by construction:
- **Non-anticipating controller.** Every 2d steps the controller copies the current state into K × R lanes (K = 6 arms, R = 8 rollouts). Each lane runs its arm for one segment and then B\* until the horizon. The controller executes the arm with the best mean rollout value. The rollouts use **fresh nuisance**: candidate sets and fantasy draws independent of the evaluated trajectory's, so nothing about the realized future leaks into the choice. This is one-step policy improvement over B\* (Bertsekas' rollout algorithm). Its gain over B\* is a *lower* bound on what optimal switching could gain, and an unbiased estimate of what this rule gains.
- **Two value models.**
  - *privileged*: rollouts query the true f, and the value is the final regret. This is an upper reference: switching with task knowledge.
  - *belief*: rollouts observe y sampled sequentially from the PFN's posterior predictive, and the value is the best fantasized y. It needs no task knowledge. **This is what a policy amortized over the prior could learn**, since the PFN's predictive *is* the prior's posterior predictive.
- **Null control (equal arms).** The same privileged controller choosing among 6 copies of B\*. Each copy is restricted to its own random half of the candidates (`B*@k`), redrawn per step and replicate, so the arm identities carry no information. Its gain over the static `B*@0` must be 0. A CI excluding 0 means the design leaks, and the experiment is INVALID.

**Setup.**
- **Tasks.** BNN-prior functions at d ∈ {2, 4, 6}. The portfolio and B\* come from `static_tune` (tuning split 0). The pilot uses split 2 (48 tasks/d) and confirm uses split 1, so B\* is **re-measured on fresh tasks**.
- **Budget.** n_init = 2d, T = 10d, 5 decisions (segments of 2d steps). Context ≤ 72 < 100: in range for pfn_large.
- **Members.** `task.method ∈ {static, privileged, belief, equal_arms}` × d. The `static` member runs the 6 portfolio arms, `B*@0` and random search on the same tasks, all paired (common random numbers per task and step).
- **Numerics.** fp32 throughout. TF32 would flip ~7% of near-tie choices, adding chaos (not bias) to the paired differences (log §1).

**What to look for.**
- **Primary metric.** The paired gain over the static B\* on the same tasks: 1 − geometric-mean ratio of final simple regrets (log10(regret + 1e-4)), with a 95% bootstrap CI over tasks.
- **Secondary.**
  - regret curves;
  - the gains of every static arm over B\*, i.e. how flat the top is;
  - which arms the controller picks per decision and d (e.g. does it explore early and exploit late?);
  - agreement between the belief and privileged choices.
- **Power.** The MDE at 80% power from the per-task sd of the log ratio.

**Decision rules (fixed before any switching run; `analysis/bo.py::switching_verdict`).**
- *Pilot* (split 2, 48 tasks/d): GO to confirm if the null CIs include 0 at every d. The pilot's sd of the per-task log ratio sizes the confirm run, which is capped by the GPU window. The pilot's effect sizes are not used for any decision.
- *Confirm* (split 1, 400 tasks/d unless the pilot's sd says otherwise):
  - **INVALID** if any null CI excludes 0;
  - **GO** for amortizing a switching policy if the *belief* controller gains ≥ 5% over B\* with its CI above 0 at ≥ 2 of the 3 d;
  - **NO-GO** if the *privileged* controller's CI upper bound is below 10% at ≥ 2 of the 3 d. In words: even with task knowledge, a gain worth a learned policy is implausible;
  - **INCONCLUSIVE** otherwise, including "too imprecise to rule anything out".

*Revision (2026-10-21 15:20, before any switching run).* The first version declared NO-GO on a privileged point estimate below 5%. `static_tune` then showed that the per-task sd of the log10 regret ratio between two good arms is ≈ 0.55. At 400 tasks that makes the CI half-width about ±12% of gain, so a point estimate below 5% would not rule out a 10% gain. The NO-GO rule now requires the CI to exclude 10%. Only the variance from the tuning split informed this change, never a switching result. Rollouts also use a cheaper candidate set (512 Sobol + 3 × 32 local; the evaluated trajectory keeps 1024 + 3 × 64). This is what makes 400 tasks/d affordable. It can only make the controller's value estimates noisier, which dilutes the gain and does not bias it.

**Limits.**
- In-distribution only: BNN-prior tasks, noise-free.
- One-step lookahead over a fixed base. A multi-step or learned policy could do better than this controller (this is a lower bound on switching's value), but it has nothing better to learn from than the belief controller's signal.
- K = 6 arms. The rollout estimate has Monte-Carlo noise (R = 8), which dilutes, but does not bias, the gain.
- One nuisance seed.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from anytime_acq.analysis import load_snapshot, show_header
from anytime_acq.analysis.bo import final, mde_gain, paired_gain, regrets, switching_verdict, tasks_for_mde

SNAPSHOT = "results/switching/<sweep_id>"  # set after `just snapshot`
snap = load_snapshot(SNAPSHOT)
show_header(snap)

## Results

All methods ran on the same tasks (same split, task list and nuisance streams); regrets use one f\* per task.

In [ ]:
cfg = yaml.safe_load(snap.experiment_yaml)["task"]
BASE = {int(d): a for d, a in cfg["base_by_d"].items()}
ARMS = {int(d): a for d, a in cfg["arms_by_d"].items()}
d_of = snap.runs.set_index("member")["task.d"]
reg = regrets(snap.table("best"), snap.table("fstar"), d_of)
fin = final(reg)
DS = sorted(fin.d.unique())
print("tasks per (d, arm):", fin.groupby(["d", "arm"]).size().unique(), "| B*:", BASE)

### Primary comparison

Gain over the static B\* (1 − geometric-mean ratio of final regrets, paired over tasks, 95% bootstrap CI). The null (`equal_arms`) is compared with its static counterpart `B*@0`.

In [ ]:
rows = []
for d in DS:
    b = BASE[d]
    for method, ref in [("ctrl:belief", b), ("ctrl:privileged", b), ("ctrl:equal_arms", f"{b}@0")]:
        if method in set(fin.loc[fin.d == d, "arm"]):
            rows.append({"method": method, **paired_gain(fin, d, method, ref)})
primary = pd.DataFrame(rows)
primary["mde80"] = [mde_gain(s, n) for s, n in zip(primary.sd_log_ratio, primary.n, strict=True)]
primary[["d", "method", "b", "n", "gain", "gain_lo", "gain_hi", "frac_a_better", "sd_log_ratio", "mde80"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for k, (m, g) in enumerate(primary.groupby("method")):
    x = np.array([DS.index(d) for d in g.d]) + (k - 1) * 0.2
    ax.errorbar(x, g.gain, yerr=[g.gain - g.gain_lo, g.gain_hi - g.gain], fmt="o", capsize=3, label=m)
ax.axhline(0, c="k", lw=0.8); ax.axhline(0.05, c="gray", ls=":", lw=0.8)
ax.set_xticks(range(len(DS)), [f"d = {d}" for d in DS]); ax.set_ylabel("gain over B* (null: over B*@0)")
ax.legend(fontsize=8); fig.tight_layout()

### Static arms on these tasks: how flat is the top?

The portfolio arms and random search, each against B\*, on the same fresh tasks. B\* was selected on split 0, so this is also the held-out check of the selection.

In [ ]:
flat = pd.DataFrame([paired_gain(fin, d, a, BASE[d]) for d in DS for a in [*ARMS[d][1:], "random"]
                     if a in set(fin.loc[fin.d == d, "arm"])])
flat.pivot(index="a", columns="d", values="gain").round(3)

### Regret curves

In [ ]:
fig, axes = plt.subplots(1, len(DS), figsize=(5 * len(DS), 4))
for ax, d in zip(np.atleast_1d(axes), DS, strict=True):
    g = reg.loc[reg.d == d]
    curves = g.groupby(["arm", "t"]).log_regret.mean().unstack(0)
    for arm in curves.columns:
        style = dict(lw=2.2) if arm.startswith("ctrl:") else dict(lw=0.9, alpha=0.7, ls="--" if arm == "random" else "-")
        ax.plot(curves.index, 10 ** curves[arm], label=arm, **style)
    ax.set_yscale("log"); ax.set_title(f"d = {d}"); ax.set_xlabel("BO step t"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("geo-mean simple regret"); np.atleast_1d(axes)[-1].legend(fontsize=7)
fig.tight_layout()

### What the controllers choose

Share of tasks choosing each arm, per decision (decision j starts at step 2d·j). At decision 0 the state is identical for both controllers, so their agreement there compares the belief signal with the privileged one directly.

In [ ]:
dec = snap.table("decisions").assign(d=lambda x: x.member.map(d_of))
dec["method"] = dec.member.map(snap.runs.set_index("member")["task.method"])
share = dec.groupby(["d", "method", "decision"]).chosen.value_counts(normalize=True).unstack(fill_value=0)
display(share.loc[(slice(None), ["belief", "privileged"]), :].round(2))
d0 = dec.loc[dec.decision == 0].pivot_table(index=["d", "seed"], columns="method", values="chosen", aggfunc="first")
if {"belief", "privileged"} <= set(d0.columns):
    agree = (d0.belief == d0.privileged).groupby("d").mean()
    print("decision-0 agreement belief vs privileged:", agree.round(2).to_dict())

### Power

The per-task sd of the log10 regret ratio sets the number of tasks for an MDE of 5% (80% power). In the pilot this sizes the confirm run.

In [ ]:
primary.assign(n_for_mde5=[tasks_for_mde(s, 0.05) for s in primary.sd_log_ratio])[["d", "method", "sd_log_ratio", "n", "mde80", "n_for_mde5"]].round(3)

## Observations

_Filled from the outputs above._

In [ ]:
def by(method):
    return primary.loc[primary.method == method].to_dict("records")

print("pre-registered verdict:", switching_verdict(by("ctrl:belief"), by("ctrl:privileged"), by("ctrl:equal_arms")))

## Verdict

_From the rule above; recorded with `just decide`._